# Inspect a saved Fraise run

Install notebook tools with `uv sync --extra notebooks`. Select a run directory below. The shared loader preserves its archived configuration, tokenizer, and weight tying. Set `FRAISE_DATA_DIR` when launching outside the project.


In [ ]:
from fraise.paths import data_root
from fraise.inference.artifacts import list_model_runs, load_run
from fraise.inference.scoring import score_text, top_next_tokens
from fraise.inference.generation import generate
from fraise.inference.benchmark import benchmark_generation

runs = list_model_runs(data_root() / "models")
for number, path in enumerate(runs, 1):
    print(number, path)


In [ ]:
RUN_DIRECTORY = runs[0]  # Select the run you want to inspect.
CHECKPOINT = None  # Optional checkpoint filename; otherwise load the final model or latest step.
run = load_run(RUN_DIRECTORY, checkpoint=CHECKPOINT)
model, tokenizer = run.model, run.tokenizer
model.compile()
print(run.path, run.config)
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))


In [ ]:
TEST_TEXT = "We explore efficient language modeling with compressed representations."
print(score_text(model, tokenizer, TEST_TEXT))
for token_id, probability, text in top_next_tokens(model, tokenizer, "The future of small language models is"):
    print(token_id, probability, repr(text))


In [ ]:
PROMPT = "The future of small language models is"
text, ids = generate(model, tokenizer, PROMPT, max_new_tokens=48, temperature=1.0)
print(text)


## Full-context decoding throughput

This measures greedy generation without a KV cache. Warmup runs are excluded; fixed-length timed runs continue through EOS.


In [ ]:
print(benchmark_generation(model, tokenizer, PROMPT, max_new_tokens=32, warmup_runs=1, timed_runs=3))
